# Week 10 · Class 5 — Independent Capstone
## Historical Production Data Quality Pipeline

**Assessment notebook — not a guided teaching lesson.** Produce a self-contained, reproducible analytic report. Include your own explanations, assumptions, code outputs, and findings. This notebook must call and demonstrate the functions you implement in `src/production_pipeline.py`.

**Important:** Use the full project brief as the source of truth. Do not delete the original dataset or change input values.

## 1. Problem statement and objectives
Write 1–2 paragraphs describing why missing 6-hour observations, duplicates, incorrect measurements, and changing sensor reliability matter. State the required outputs and your policy for zeros versus missing readings.

### 1. Problem Statement and Objectives

Oil wells produce data at different times, and this data helps us understand how each well is performing. Sometimes, some readings may be missing, especially when a sensor fails to send data every six hours. There may also be duplicate records or incorrect values, such as negative numbers or text in numeric columns. These problems can affect production calculations and make the results unreliable. It is therefore important to check and clean the data before using it for analysis.

The aim of this project is to build a Python pipeline that checks production data, removes duplicate records, handles invalid values, and arranges well readings at six-hour intervals. It also calculates liquid production and water cut, creates daily production summaries, and produces reports showing data quality and rejected records. In this project, zero is treated as a valid reading, while missing values remain missing. Missing oil, water, and gas values are not filled with estimated values. Only one internal missing pressure reading can be interpolated, without filling missing values at the beginning or end of the data.

In [9]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src" / "production_pipeline.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Project folder not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.production_pipeline import (
    ingest_and_validate,
    repair_and_align,
    summarize_daily,
    run_pipeline
)

INPUT = PROJECT_ROOT / "data" / "historical_production_capstone.csv"
OUTPUT = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Input present:", INPUT.exists())

Project root: c:\Users\ismai\Documents\Charlotte2026\Module 2\Charlotte-Week10-python\day5_capstone
Input present: True


## 2. Data ingestion and schema validation
Show raw row count, column names, data types, sample rows and invalid timestamps. Explain why timestamp parsing uses `errors="coerce"`.

In [12]:
# TODO: inspect the raw CSV and demonstrate ingest_and_validate(INPUT)
raw_df = pd.read_csv(INPUT)

print("Raw row count:", len(raw_df))

print("\nColumn names:")
print(raw_df.columns.tolist())

print("\nData types:")
print(raw_df.dtypes)

print("\nSample rows:")
display(raw_df.head())

valid_rows, rejected_rows = ingest_and_validate(INPUT)

print("\nValid timestamp rows:", len(valid_rows))
print("Rejected timestamp rows:", len(rejected_rows))

print("\nRejected records:")
display(rejected_rows)

Raw row count: 47

Column names:
['well_id', 'timestamp', 'oil_bbl', 'water_bbl', 'gas_mscf', 'pressure_psi', 'status', 'ingested_at']

Data types:
well_id             str
timestamp           str
oil_bbl         float64
water_bbl         int64
gas_mscf            str
pressure_psi        str
status              str
ingested_at         str
dtype: object

Sample rows:


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at
0,WELL-21,2026-04-06 18:00,77.0,22,151,2839,ACTIVE,2026-04-06 18:04
1,WELL-21,2026-04-08 00:00,68.0,24,161,2904,ACTIVE,2026-04-08 00:04
2,WELL-21,2026-04-09 00:00,68.0,22,169,2956,ACTIVE,2026-04-09 00:04
3,WELL-22,2026-04-06 00:00,78.0,24,152,2960,ACTIVE,2026-04-06 00:04
4,WELL-22,2026-04-07 00:00,78.0,25,160,not_available,ACTIVE,2026-04-07 00:04



Valid timestamp rows: 46
Rejected timestamp rows: 1

Rejected records:


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at,rejection_reason
34,WELL-22,2026-04-99 12:00,72.0,22,150,2960,ACTIVE,2026-04-10 01:00,Invalid timestamp


The errors="coerce" option converts invalid timestamps to NaT instead of stopping the program with an error. 
This helps us identify invalid timestamps and separate rejected records from valid records while keeping the original source data for review.

## 3. Deduplication, alignment, and cleaning
Show how latest-ingestion duplicate precedence works. Explain how missing *timestamps* differ from missing *values*. Display a before-and-after example for a chosen well. Explain how you prevent false oil production totals.

In [14]:
# TODO: apply repair_and_align; show per-well sample rows, missingness counts, and quality events
# Apply repair and alignment
cleaned_rows, quality_report = repair_and_align(valid_rows)

# Show duplicate records before cleaning
duplicate_rows = valid_rows[
    valid_rows.duplicated(
        subset=["well_id", "timestamp"],
        keep=False
    )
].sort_values(["well_id", "timestamp", "ingested_at"])

print("Duplicate records before cleaning:")
display(duplicate_rows)

# Show before-and-after example
chosen_well = duplicate_rows.iloc[0]["well_id"]
chosen_time = duplicate_rows.iloc[0]["timestamp"]

print("Before cleaning:")
display(valid_rows[
    (valid_rows["well_id"] == chosen_well)
    & (valid_rows["timestamp"] == chosen_time)
])

print("After cleaning:")
display(cleaned_rows[
    (cleaned_rows["well_id"] == chosen_well)
    & (cleaned_rows["timestamp"] == chosen_time)
])

# Show sample rows for each well
print("Sample rows per well:")
display(cleaned_rows.groupby("well_id").head(2))

# Show missing values
print("Missing values:")
display(cleaned_rows[
    ["oil_bbl", "water_bbl", "gas_mscf", "pressure_psi"]
].isna().sum())

# Show quality events
print("Quality report:")
display(quality_report)

Duplicate records before cleaning:


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at
35,WELL-21,2026-04-06 06:00:00,71.0,23,147,2813,ACTIVE,2026-04-06 06:04
22,WELL-21,2026-04-06 06:00:00,76.0,23,147,2813,ACTIVE,2026-04-06 06:24
9,WELL-23,2026-04-08 12:00:00,94.0,27,179,3250,ACTIVE,2026-04-08 12:04
46,WELL-23,2026-04-08 12:00:00,98.0,27,179,3250,ACTIVE,2026-04-08 12:24


Before cleaning:


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at
22,WELL-21,2026-04-06 06:00:00,76.0,23,147,2813,ACTIVE,2026-04-06 06:24
35,WELL-21,2026-04-06 06:00:00,71.0,23,147,2813,ACTIVE,2026-04-06 06:04


After cleaning:


,timestamp,well_id,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at,inserted_slot,liquid_bbl,water_cut_pct
1,2026-04-06 06:00:00,WELL-21,76.0,23.0,147.0,2813.0,ACTIVE,2026-04-06 06:24,False,99.0,23.232323


... (2 outputs left out)

Missing values:


oil_bbl         6
water_bbl       5
gas_mscf        5
pressure_psi    0
dtype: int64

Quality report:


,duplicate_rows_removed,inserted_slots,missing_oil_bbl,missing_water_bbl,missing_gas_mscf,missing_pressure_psi
0,2,4,6,5,5,0


For WELL-21 at 2026-04-06 06:00, two records were found with different ingestion times. 
The record ingested at 06:24 with 76 barrels of oil was retained instead of the earlier record with 71 barrels. 
This prevents the same observation from being counted twice.

A missing six-hour timestamp means that an expected observation is not recorded, while a missing value means the record exists but its measurement is missing or invalid. 
Missing oil values are not replaced with zero because doing so could give incorrect production totals.

## 4. Daily resampling / aggregation
Report daily oil, water and gas totals, observation completeness, and incomplete days. Explain why sum(min_count=1) protects all-missing groups.

In [16]:
# TODO: call summarize_daily and display a well/day summary
daily_summary = summarize_daily(cleaned_rows)

print("Daily production summary:")
display(daily_summary)

completeness_columns = [
    "oil_complete",
    "water_complete",
    "gas_complete"
]

incomplete_days = daily_summary.loc[
    ~daily_summary[completeness_columns].all(axis=1)
]

print("Number of incomplete days:", len(incomplete_days))

print("\nIncomplete days:")
display(incomplete_days)

Daily production summary:


,well_id,date,oil_bbl,water_bbl,gas_mscf,expected_slots,oil_valid_readings,water_valid_readings,gas_valid_readings,oil_complete,water_complete,gas_complete
0,WELL-21,2026-04-06,221.0,91.0,592.0,4,3,4,4,False,True,True
1,WELL-21,2026-04-07,219.0,68.0,469.0,4,3,3,3,False,False,False
2,WELL-21,2026-04-08,290.0,71.0,656.0,4,4,3,4,True,False,True
3,WELL-21,2026-04-09,290.0,91.0,688.0,4,4,4,4,True,True,True
4,WELL-22,2026-04-06,330.0,99.0,620.0,4,4,4,4,True,True,True
5,WELL-22,2026-04-07,243.0,75.0,486.0,4,4,4,4,True,True,True
6,WELL-22,2026-04-08,159.0,76.0,512.0,4,2,3,3,False,False,False
7,WELL-22,2026-04-09,330.0,99.0,716.0,4,4,4,4,True,True,True
8,WELL-23,2026-04-06,273.0,81.0,483.0,4,3,3,3,False,False,False
9,WELL-23,2026-04-07,370.0,108.0,680.0,4,4,4,4,True,True,True


Number of incomplete days: 7

Incomplete days:


,well_id,date,oil_bbl,water_bbl,gas_mscf,expected_slots,oil_valid_readings,water_valid_readings,gas_valid_readings,oil_complete,water_complete,gas_complete
0,WELL-21,2026-04-06,221.0,91.0,592.0,4,3,4,4,False,True,True
1,WELL-21,2026-04-07,219.0,68.0,469.0,4,3,3,3,False,False,False
2,WELL-21,2026-04-08,290.0,71.0,656.0,4,4,3,4,True,False,True
6,WELL-22,2026-04-08,159.0,76.0,512.0,4,2,3,3,False,False,False
8,WELL-23,2026-04-06,273.0,81.0,483.0,4,3,3,3,False,False,False
10,WELL-23,2026-04-08,374.0,109.0,537.0,4,4,4,3,True,True,False
11,WELL-23,2026-04-09,282.0,81.0,561.0,4,3,3,3,False,False,False


The daily summary contains 12 records, with four expected observations per well each day. Out of these, 7 days were incomplete and 5 days were complete. 
The completeness columns help identify which production measurements are missing for each well and date.

The `sum(min_count=1)` method prevents a group with all missing values from being reported as zero. Instead, it returns NaN, which helps avoid false production totals.
A genuine zero remains a valid measurement.

## 5. Validation evidence and interpretation
Provide results of tests you wrote, expected-vs-actual checks for row counts, and at least three operational insights or limitations. Note that the data are synthetic.

In [17]:
# TODO: inspect the raw CSV and demonstrate ingest_and_validate(INPUT)
# Expected versus actual results

expected_results = {
    "Raw rows": (47, len(raw_df)),
    "Valid timestamp rows": (46, len(valid_rows)),
    "Rejected timestamp rows": (1, len(rejected_rows)),
    "Duplicate rows removed": (
        2, int(quality_report.loc[0, "duplicate_rows_removed"])
    ),
    "Inserted slots": (
        4, int(quality_report.loc[0, "inserted_slots"])
    ),
    "Cleaned rows": (48, len(cleaned_rows)),
    "Daily summary rows": (12, len(daily_summary))
}

validation_results = pd.DataFrame([
    {
        "Check": name,
        "Expected": expected,
        "Actual": actual,
        "Passed": expected == actual
    }
    for name, (expected, actual) in expected_results.items()
])

display(validation_results)

print("All validation checks passed:",
      validation_results["Passed"].all())

print("Pytest results: 3 passed")
raw_df = pd.read_csv(INPUT)

print("Raw row count:", len(raw_df))

print("\nColumn names:")
print(raw_df.columns.tolist())

print("\nData types:")
print(raw_df.dtypes)

print("\nSample rows:")
display(raw_df.head())

valid_rows, rejected_rows = ingest_and_validate(INPUT)

print("\nValid timestamp rows:", len(valid_rows))
print("Rejected timestamp rows:", len(rejected_rows))

print("\nRejected records:")
display(rejected_rows)

,Check,Expected,Actual,Passed
0,Raw rows,47,47,True
1,Valid timestamp rows,46,46,True
2,Rejected timestamp rows,1,1,True
3,Duplicate rows removed,2,2,True
4,Inserted slots,4,4,True
5,Cleaned rows,48,48,True
6,Daily summary rows,12,12,True


All validation checks passed: True
Pytest results: 3 passed
Raw row count: 47

Column names:
['well_id', 'timestamp', 'oil_bbl', 'water_bbl', 'gas_mscf', 'pressure_psi', 'status', 'ingested_at']

Data types:
well_id             str
timestamp           str
oil_bbl         float64
water_bbl         int64
gas_mscf            str
pressure_psi        str
status              str
ingested_at         str
dtype: object

Sample rows:


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at
0,WELL-21,2026-04-06 18:00,77.0,22,151,2839,ACTIVE,2026-04-06 18:04
1,WELL-21,2026-04-08 00:00,68.0,24,161,2904,ACTIVE,2026-04-08 00:04
2,WELL-21,2026-04-09 00:00,68.0,22,169,2956,ACTIVE,2026-04-09 00:04
3,WELL-22,2026-04-06 00:00,78.0,24,152,2960,ACTIVE,2026-04-06 00:04
4,WELL-22,2026-04-07 00:00,78.0,25,160,not_available,ACTIVE,2026-04-07 00:04



Valid timestamp rows: 46
Rejected timestamp rows: 1

Rejected records:


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at,rejection_reason
34,WELL-22,2026-04-99 12:00,72.0,22,150,2960,ACTIVE,2026-04-10 01:00,Invalid timestamp


### Operational Insights and Limitations

1. Seven out of the twelve daily well summaries were incomplete. This shows that some expected production measurements were missing.
2. Two duplicate records were removed by keeping the latest ingested record. This prevents the same observation from being counted twice.
3. The dataset contained missing measurements: 6 oil values, 5 water values and 5 gas values. Missing production values were not replaced with zero because this could give incorrect totals.

The dataset used in this project is synthetic and does not represent actual oilfield production. 
Also, pressure interpolation is limited to one internal missing observation. The results should therefore be used for data-quality practice and not as a basis for real operational decisions.

In [18]:
# TODO: run_pipeline(INPUT, OUTPUT); print the names of the four output files
# Run the complete production pipeline
run_pipeline(INPUT, OUTPUT)

# Print the names of the four output files
output_files = sorted(
    file.name for file in OUTPUT.glob("*.csv")
)

print("Generated output files:")
for file_name in output_files:
    print(file_name)

Generated output files:
cleaned_production.csv
daily_production_summary.csv
data_quality_report.csv
rejected_source_records.csv


## 6. Conclusions and reproducibility
Explain your cleaning decisions, their impact on interpretation, and how another analyst can rerun the pipeline. Include a link to the GitHub Pull Request in your README (do not hard-code a private token).

### Conclusions and Reproducibility

In this project, I cleaned the historical production data by checking timestamps, removing duplicate records, converting invalid numeric values and negative measurements to missing values, and arranging the readings at six-hour intervals. When duplicate records had the same well ID and timestamp, I kept the record with the latest ingestion time. I also kept genuine zero values because zero is a valid measurement. Missing oil, water and gas values were not filled with estimated values, while pressure interpolation was limited to one internal missing observation.

These cleaning decisions improved the reliability of the daily production summaries. Keeping missing values instead of replacing them with zero helps prevent false production totals. The completeness indicators also make it easier to identify days with missing measurements before using the results for further analysis.

Another analyst can reproduce this work by using the project files and installing the required packages from `requirements.txt`. They can run the supplied tests and execute `run_pipeline()` using the original CSV file. The pipeline will generate four CSV reports in the outputs folder. The dataset used in this project is synthetic, so the findings are for learning and data-quality analysis rather than actual field decisions.